# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The action queue prioritizes content items using the observed baseline action score from ML-07. Higher scores indicate higher rule-based priority for human review.

Reason codes explain why an item was prioritized, such as a page-one opportunity, low CTR on a visible page, stale visible page, thin visible page, or general refresh review.

The queue is intended as decision-support. The score does not prove that a page will improve after an action, so content teams should review the underlying signals before making changes.


In [23]:
# ML-10 Section 1: Ranked actions + reason codes

import os
import numpy as np
import pandas as pd

from datasets import load_dataset

# ---------------------------------------------------------
# Load dataset from Hugging Face
# ---------------------------------------------------------
dataset = load_dataset("FlyRank/internship-starter")
df = dataset["train"].to_pandas()





print("Rows:", len(df))
print("Columns:", len(df.columns))

# ---------------------------------------------------------
# Recreate the ML-07 baseline action score
# ---------------------------------------------------------

def percentile_rank(series):
    return series.rank(
        pct=True,
        method="average"
    ).fillna(0)


impressions = df["impressions_90d"].fillna(
    df["impressions_90d"].median()
)

days_update = df["days_since_last_update"].fillna(
    df["days_since_last_update"].median()
)

avg_position = df["avg_position"].fillna(
    df["avg_position"].median()
)

word_count = df["word_count"].fillna(
    df["word_count"].median()
)

ctr = df["ctr"].fillna(
    df["ctr"].median()
)

# Visibility
visibility_score = percentile_rank(
    np.log1p(impressions)
)

# Freshness risk
freshness_risk_score = percentile_rank(
    days_update
)

# Position opportunity
position_component = (
    1 - percentile_rank(
        avg_position.clip(lower=1, upper=50)
    )
)

position_opportunity_score = (
    position_component
    * visibility_score
    * (avg_position > 0).astype(int)
)

# Depth gap
depth_gap_score = (
    1 - percentile_rank(word_count)
) * visibility_score

# CTR opportunity
low_ctr_score = (
    1 - percentile_rank(ctr)
) * visibility_score

# ---------------------------------------------------------
# Baseline action score
# Same core rule used in ML-07
# ---------------------------------------------------------

baseline_score = (
    0.35 * visibility_score
    + 0.25 * freshness_risk_score
    + 0.25 * position_opportunity_score
    + 0.10 * depth_gap_score
    + 0.05 * low_ctr_score
)

# ---------------------------------------------------------
# Reason codes
# ---------------------------------------------------------

def get_reason_codes(row):

    reasons = []

    if row["visibility_score"] >= 0.80:
        if row["days_since_last_update"] >= 90:
            reasons.append("stale_visible_page")

        if pd.notna(row["word_count"]) and row["word_count"] <= 500:
            reasons.append("thin_visible_page")

        if row["avg_position"] <= 10:
            reasons.append("page_one_opportunity")

        if row["ctr"] < 0.30:
            reasons.append("low_ctr_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


queue = pd.DataFrame({
    "content_id": df["content_id"],
    "baseline_action_score": baseline_score,
    "visibility_score": visibility_score,
    "freshness_risk_score": freshness_risk_score,
    "position_opportunity_score": position_opportunity_score,
    "depth_gap_score": depth_gap_score,
    "days_since_last_update": df["days_since_last_update"],
    "avg_position": df["avg_position"],
    "ctr": df["ctr"],
    "word_count": df["word_count"]
})

queue["reason_codes"] = queue.apply(
    get_reason_codes,
    axis=1
)

# ---------------------------------------------------------
# Rank the queue
# ---------------------------------------------------------

queue = queue.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

queue["baseline_rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# Suggested action
# ---------------------------------------------------------

def suggested_action(reason):

    if "low_ctr_visible_page" in reason:
        return "refresh_and_review_ctr"

    if "page_one_opportunity" in reason:
        return "refresh_and_review_position"

    if "stale_visible_page" in reason:
        return "refresh_stale_content"

    if "thin_visible_page" in reason:
        return "review_content_depth"

    return "monitor"


queue["suggested_action"] = queue["reason_codes"].apply(
    suggested_action
)

# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("\nRanked action queue created.")

print("Rows:", len(queue))
print("Top score:", round(queue["baseline_action_score"].max(), 4))
print("Median score:", round(queue["baseline_action_score"].median(), 4))

print("\nTop 20 ranked actions:")

display(
    queue[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "reason_codes",
            "suggested_action"
        ]
    ].head(20)
)

Rows: 30000
Columns: 53

Ranked action queue created.
Rows: 30000
Top score: 0.883
Median score: 0.3891

Top 20 ranked actions:


,baseline_rank,content_id,baseline_action_score,reason_codes,suggested_action
0,1,content_681d93f6924d,0.882995,stale_visible_page|page_one_opportunity|low_ct...,refresh_and_review_ctr
1,2,content_6ac3ab740bbf,0.874176,stale_visible_page|page_one_opportunity|low_ct...,refresh_and_review_ctr
2,3,content_399f4eed93b9,0.870747,stale_visible_page|page_one_opportunity,refresh_and_review_position
3,4,content_03d2673b2553,0.864544,stale_visible_page|page_one_opportunity,refresh_and_review_position
4,5,content_a56a1c89ab0a,0.863361,stale_visible_page|page_one_opportunity|low_ct...,refresh_and_review_ctr
5,6,content_e6e15ac13287,0.862673,stale_visible_page|page_one_opportunity,refresh_and_review_position
6,7,content_e1501cdeca69,0.860817,stale_visible_page|page_one_opportunity,refresh_and_review_position
7,8,content_5184b85dc6dd,0.860141,stale_visible_page|page_one_opportunity,refresh_and_review_position
8,9,content_3d94572c3a35,0.858305,stale_visible_page|page_one_opportunity|low_ct...,refresh_and_review_ctr
9,10,content_fea6a0d13b4a,0.858202,stale_visible_page|page_one_opportunity|low_ct...,refresh_and_review_ctr


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*
## answer
## 2. Intended use and limits

The ranked queue is intended for content and SEO teams to decide which content items to review first. It provides decision-support by using the observed ML-07 baseline action score and related content signals.

The queue is not intended to automatically publish, delete, rewrite, or redirect content. A higher score means higher priority for human review; it does not prove that making a change will improve traffic, rankings, or CTR.

The recommendations may become less useful when the underlying data is missing, outdated, or changes over time. A human should review the page, search intent, current performance, and business context before taking action.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 2: Intended use and limits
# Basic checks that the action queue contains usable recommendations.

print("Total queue items:", len(queue))

print(
    "Missing baseline scores:",
    queue["baseline_action_score"].isna().sum()
)

print(
    "Missing reason codes:",
    queue["reason_codes"].isna().sum()
)

print(
    "Missing suggested actions:",
    queue["suggested_action"].isna().sum()
)

print("\nSuggested action counts:")
display(
    queue["suggested_action"].value_counts()
)

Total queue items: 30000
Missing baseline scores: 0
Missing reason codes: 0
Missing suggested actions: 0

Suggested action counts:


,count
suggested_action,
monitor,24307
refresh_and_review_ctr,3719
refresh_and_review_position,1544
refresh_stale_content,430


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

#ANSWER
## 3. Human review + the no-go list

Before taking action, a content reviewer should check the page's current search intent, relevance, accuracy, content quality, and recent performance. The reviewer should also confirm that the reason code matches the underlying signals and that there is a real opportunity to make a useful change.

The following actions should not be automated from this queue: publishing or rewriting content, deleting pages, redirecting URLs, changing canonical URLs, making claims about business or legal content, or making changes that could affect users without human approval.

The queue identifies items for review; the final decision remains with a human reviewer.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 3: Human review + no-go list
# Check that every queue item has a reason and an action.

review_check = queue[
    [
        "content_id",
        "baseline_action_score",
        "reason_codes",
        "suggested_action"
    ]
].copy()

print("Items requiring human review:", len(review_check))

print(
    "Items with a reason code:",
    review_check["reason_codes"].notna().sum()
)

print(
    "Items with a suggested action:",
    review_check["suggested_action"].notna().sum()
)

print("\nReason-code distribution:")
display(
    review_check["reason_codes"]
    .value_counts()
    .head(10)
)

Items requiring human review: 30000
Items with a reason code: 30000
Items with a suggested action: 30000

Reason-code distribution:


,count
reason_codes,
general_refresh_review,24307
page_one_opportunity|low_ctr_visible_page,1186
page_one_opportunity,1038
stale_visible_page|low_ctr_visible_page,915
low_ctr_visible_page,854
stale_visible_page|page_one_opportunity|low_ctr_visible_page,764
stale_visible_page|page_one_opportunity,506
stale_visible_page,430


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

ANSWER

## 4. Monitoring / retrain triggers

The recommendations should be monitored because the underlying content and search signals can change over time. The queue should be regenerated when important input distributions change, when the share of high-priority items changes substantially, or when the action outcomes no longer match the assumptions used to create the baseline score.

A practical trigger is to investigate when the share of high-priority items changes materially from the current queue or when important inputs such as impressions, CTR, average position, content age, or word count show a large distribution shift.

The baseline scoring rules should be reviewed or retrained when these changes persist and the recommendations are no longer useful to content reviewers. These triggers are monitoring rules, not proof that the system has failed.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 4: Monitoring / retrain triggers
# Record the current queue as a baseline for future monitoring.

monitoring_baseline = {
    "queue_size": len(queue),

    "high_priority_share": (
        queue["baseline_action_score"] >= 0.80
    ).mean(),

    "mean_action_score": (
        queue["baseline_action_score"].mean()
    ),

    "median_action_score": (
        queue["baseline_action_score"].median()
    ),

    "top_score": (
        queue["baseline_action_score"].max()
    )
}

print("Monitoring baseline:")

for key, value in monitoring_baseline.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Monitoring baseline:
queue_size: 30000
high_priority_share: 0.0109
mean_action_score: 0.3941
median_action_score: 0.3891
top_score: 0.8830


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

ANSWER
## 5. Exports for the paper

The ranked action queue is exported so that the paper can reuse the observed ML-07 baseline ranking and its reason codes. The exported file contains the rank, content identifier, baseline action score, reason codes, and suggested action.

The export is an analysis artifact for the project and does not contain client names, URLs, or private queries.


In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 5: Export the ranked action queue

from pathlib import Path

# ------------------------------------------------------------
# 1. Create the required output directory
# ------------------------------------------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Select the queue fields needed for the paper
# ------------------------------------------------------------

paper_queue = queue[
    [
        "baseline_rank",
        "content_id",
        "baseline_action_score",
        "reason_codes",
        "suggested_action"
    ]
].copy()


# ------------------------------------------------------------
# 3. Export the complete ranked queue
# ------------------------------------------------------------

output_file = output_dir / "ml10_ranked_action_queue.csv"

paper_queue.to_csv(
    output_file,
    index=False
)


# ------------------------------------------------------------
# 4. Verify the exported file
# ------------------------------------------------------------

print("Export complete.")
print("File:", output_file)
print("Rows:", len(paper_queue))
print("Columns:", len(paper_queue.columns))

print("\nExported columns:")
print(paper_queue.columns.tolist())

print("\nFile exists:", output_file.exists())
print("File size (bytes):", output_file.stat().st_size)

Export complete.
File: work/outputs/ml10_ranked_action_queue.csv
Rows: 30000
Columns: 5

Exported columns:
['baseline_rank', 'content_id', 'baseline_action_score', 'reason_codes', 'suggested_action']

File exists: True
File size (bytes): 2451485


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.